Paths

In [2]:
DATA_DIR = Path("standardized_exports")
RESULTS_DIR = Path("results")

RESULTS_DIR.mkdir(exist_ok=True)

Horizons

In [3]:
horizons_short = {

    # Traffic (1 hour)
    "std_traffic_small": 12,
    "std_traffic_medium": 12,
    "std_traffic_large": 12,

    # Energy (24 hours)
    "std_energy_small": 96,
    "std_energy_medium": 96,
    "std_energy_large": 96,

    # Favorita (7 days)
    "std_favorita_small_train": 7,
    "std_favorita_medium_train": 7,
    "std_favorita_large_train": 7,

    "std_favorita_small_transactions": 7,
    "std_favorita_medium_transactions": 7,
    "std_favorita_large_transactions": 7,

    # Agricultural (1 year)
    "std_agri_small": 1,
    "std_agri_medium": 1,
    "std_agri_large": 1,

    # M4 Finance
    "std_m4_daily_small": 5,
    "std_m4_daily_medium": 5,
    "std_m4_daily_large": 5,

    "std_m4_hourly_small": 5,
    "std_m4_hourly_medium": 5,
    "std_m4_hourly_large": 5,

    "std_m4_weekly_small": 5,
    "std_m4_weekly_medium": 5,
    "std_m4_weekly_large": 5,

    "std_m4_monthly_small": 5,
    "std_m4_monthly_medium": 5,
    "std_m4_monthly_large": 5,

    "std_m4_quarterly_small": 5,
    "std_m4_quarterly_medium": 5,
    "std_m4_quarterly_large": 5,

    "std_m4_yearly_small": 5,
    "std_m4_yearly_medium": 5,
    "std_m4_yearly_large": 5,
}

In [4]:
horizons_medium = {

    # Traffic (2 hours)
    "std_traffic_small": 24,
    "std_traffic_medium": 24,
    "std_traffic_large": 24,

    # Energy (168 hours = 1 week)
    "std_energy_small": 336,
    "std_energy_medium": 336,
    "std_energy_large": 336,

    # Favorita (14 days)
    "std_favorita_small_train": 14,
    "std_favorita_medium_train": 14,
    "std_favorita_large_train": 14,

    "std_favorita_small_transactions": 14,
    "std_favorita_medium_transactions": 14,
    "std_favorita_large_transactions": 14,

    # Agricultural (3 years)
    "std_agri_small": 3,
    "std_agri_medium": 3,
    "std_agri_large": 3,

    # M4 Finance
    "std_m4_daily_small": 10,
    "std_m4_daily_medium": 10,
    "std_m4_daily_large": 10,

    "std_m4_hourly_small": 10,
    "std_m4_hourly_medium": 10,
    "std_m4_hourly_large": 10,

    "std_m4_weekly_small": 10,
    "std_m4_weekly_medium": 10,
    "std_m4_weekly_large": 10,

    "std_m4_monthly_small": 10,
    "std_m4_monthly_medium": 10,
    "std_m4_monthly_large": 10,

    "std_m4_quarterly_small": 10,
    "std_m4_quarterly_medium": 10,
    "std_m4_quarterly_large": 10,

    "std_m4_yearly_small": 10,
    "std_m4_yearly_medium": 10,
    "std_m4_yearly_large": 10,
}

In [5]:
horizons_long = {

    # Traffic (4 hours)
    "std_traffic_small": 48,
    "std_traffic_medium": 48,
    "std_traffic_large": 48,

    # Energy
    "std_energy_small": 672,
    "std_energy_medium": 672,
    "std_energy_large": 672,

    # Favorita (28 days)
    "std_favorita_small_train": 28,
    "std_favorita_medium_train": 28,
    "std_favorita_large_train": 28,

    "std_favorita_small_transactions": 28,
    "std_favorita_medium_transactions": 28,
    "std_favorita_large_transactions": 28,

    # Agricultural (5 years)
    "std_agri_small": 5,
    "std_agri_medium": 5,
    "std_agri_large": 5,

    # M4 Finance
    "std_m4_daily_small": 20,
    "std_m4_daily_medium": 20,
    "std_m4_daily_large": 20,

    "std_m4_hourly_small": 20,
    "std_m4_hourly_medium": 20,
    "std_m4_hourly_large": 20,

    "std_m4_weekly_small": 20,
    "std_m4_weekly_medium": 20,
    "std_m4_weekly_large": 20,

    "std_m4_monthly_small": 20,
    "std_m4_monthly_medium": 20,
    "std_m4_monthly_large": 20,

    "std_m4_quarterly_small": 20,
    "std_m4_quarterly_medium": 20,
    "std_m4_quarterly_large": 20,

    "std_m4_yearly_small": 20,
    "std_m4_yearly_medium": 20,
    "std_m4_yearly_large": 20,
}

In [6]:
def get_horizons_info(dataset_name):

    horizons = []

    horizons.append(
        (horizons_short[dataset_name], "short")
        )

    horizons.append(
        (horizons_medium[dataset_name], "medium")
    )

    horizons.append(
        (horizons_long[dataset_name], "long")
    )

    return horizons

Metrics

In [7]:
def smape(y_true, y_pred):

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(y_true[mask] - y_pred[mask])
            / denominator[mask]
        )
        * 100
    )

def evaluate(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "SMAPE": smape(y_true, y_pred)
    }

Benchmarker

In [8]:
def benchmark_dataset(
    df,
    horizon,
    forecast_function,
    model_name
):

    series_metrics = []

    start = time.time()

    for uid, group in df.groupby(
        "unique_id",
        sort=False
    ):

        y = group["y"].to_numpy()

        if len(y) <= horizon:
            continue

        train = y[:-horizon]
        test = y[-horizon:]

        pred = forecast_function(
            train,
            horizon
        )

        metrics = evaluate(
            test,
            pred
        )

        series_metrics.append(metrics)

    runtime = time.time() - start

    if len(series_metrics) == 0:
        return None

    metrics = pd.DataFrame(
        series_metrics
    ).mean()

    return {
        "model": model_name,
        "n_series": df["unique_id"].nunique(),
        "n_rows": len(df),
        "horizon": horizon,
        "runtime_seconds": runtime,
        **metrics.to_dict()
    }

In [9]:
def run_benchmark(
    model_name,
    forecast_function,
    output_file
):

    results = []

    for file in DATA_DIR.glob("*.csv"):

        dataset_name = file.stem

        try:
            dataset_horizons = get_horizons_info(dataset_name)

        except KeyError:
            print(
                f"Skipping {dataset_name}: no horizon defined"
            )
            continue

        print(
            f"\nRunning {model_name} on {dataset_name}"
        )

        df = pd.read_csv(
            file,
            low_memory=False
        )

        start = time.time()

        for horizon, horizon_type in dataset_horizons:

            result = benchmark_dataset(
                df=df,
                horizon=horizon,
                forecast_function=forecast_function,
                model_name=model_name
            )

            if result:

                result["dataset"] = dataset_name
                result["horizon_type"] = horizon_type

                results.append(result)

                pd.DataFrame(results).to_csv(
                    output_file,
                    index=False
                )

        print(
                f"Completed {dataset_name}"
                f" in {time.time()-start:.1f}s"
            )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type

            results.append(result)

            pd.DataFrame(results).to_csv(
                output_file,
                index=False
            )

        del df
        gc.collect()

    return pd.DataFrame(results)

In [10]:
season_lengths = {
    "std_agri_small": 12,
    "std_agri_medium": 12,
    "std_agri_large": 12,

    "std_energy_small": 24,
    "std_energy_medium": 24,
    "std_energy_large": 24,

    "std_traffic_small": 168,
    "std_traffic_medium": 168,
    "std_traffic_large": 168,

    "std_weather_small": 24,
    "std_weather_medium": 24,
    "std_weather_large": 24,

    "std_exchange_small": 5,
    "std_exchange_medium": 5,
    "std_exchange_large": 5,

    "std_metr_small": 288,
    "std_metr_medium": 288,
    "std_metr_large": 288
}

In [11]:
def get_available_datasets():

    return sorted(
        [
            file.stem
            for file in DATA_DIR.glob("*.csv")
        ]
    )

In [12]:
def get_dataset_freq(dataset_name):

    if dataset_name not in dataset_freqs:
        raise ValueError(
            f"No frequency defined for {dataset_name}"
        )

    return dataset_freqs[dataset_name]

In [13]:
def prepare_neuralforecast_data(df):

    required_columns = [
        "unique_id",
        "ds",
        "y"
    ]

    missing = [
        c
        for c in required_columns
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}"
        )

    df = df[
        required_columns
    ].copy()

    df["y"] = pd.to_numeric(
        df["y"],
        errors="coerce"
    )

    df = df.dropna()

    return df

In [14]:
def convert_ds(df, freq):

    df = df.copy()

    if freq == "YS":

        df["ds"] = pd.to_datetime(
            df["ds"].astype(str),
            format="%Y"
        )

    else:

        df["ds"] = pd.to_datetime(
            df["ds"],
            errors="coerce"
        )

    df = df.dropna(
        subset=["ds"]
    )

    return df

In [15]:
dataset_freqs = {

    # Agriculture
    "std_agri_small": "YS",
    "std_agri_medium": "YS",
    "std_agri_large": "YS",

    # Energy
    "std_energy_small": "H",
    "std_energy_medium": "H",
    "std_energy_large": "H",

    # Traffic
    "std_traffic_small": "H",
    "std_traffic_medium": "H",
    "std_traffic_large": "H",

    # Favorita
    "std_favorita_small_train": "D",
    "std_favorita_medium_train": "D",
    "std_favorita_large_train": "D",

    "std_favorita_small_transactions": "D",
    "std_favorita_medium_transactions": "D",
    "std_favorita_large_transactions": "D",

    # M4 Daily
    "std_m4_daily_small": "D",
    "std_m4_daily_medium": "D",
    "std_m4_daily_large": "D",

    # M4 Hourly
    "std_m4_hourly_small": "H",
    "std_m4_hourly_medium": "H",
    "std_m4_hourly_large": "H",

    # M4 Weekly
    "std_m4_weekly_small": "W",
    "std_m4_weekly_medium": "W",
    "std_m4_weekly_large": "W",

    # M4 Monthly
    "std_m4_monthly_small": "MS",
    "std_m4_monthly_medium": "MS",
    "std_m4_monthly_large": "MS",

    # M4 Quarterly
    "std_m4_quarterly_small": "QS",
    "std_m4_quarterly_medium": "QS",
    "std_m4_quarterly_large": "QS",

    # M4 Yearly
    "std_m4_yearly_small": "YS",
    "std_m4_yearly_medium": "YS",
    "std_m4_yearly_large": "YS",
}

In [16]:
def prepare_neuralforecast_dataset(
    df,
    dataset_name
):

    df = prepare_neuralforecast_data(
        df
    )

    freq = get_dataset_freq(
        dataset_name
    )

    df = convert_ds(
        df,
        freq
    )

    df = df.sort_values(
        ["unique_id", "ds"]
    )

    return df, freq